Luca Agostini, Arwin Koen S Gorissen, Philipp Josef Nimphius

# Short notebook 2: AutoGluon (LightGBM, CatBoost, ExtraTrees) + cascade LP, blended with a temporal network
**Run:** set `DATA` to the folder with the raw data (files are searched recursively by name) and run all cells; writes `submission_short_2.csv`. Needs `Historical_*`, `Synthetic_water_value_*`, `Constraint_*`, `Unit_commitment_decisions.csv`, `Tokke_Vinje_topology.yaml`, `prediction_mapping.csv`, `sample_submission.csv`. Uses one GPU (CUDA) if present, otherwise the CPU; no intermediate files.

Python packages: numpy, pandas, scipy, pyyaml, torch, autogluon.tabular[lightgbm,catboost]==1.6.3.

In [ ]:
import numpy as np, pandas as pd, yaml
import torch, torch.nn as nn
import scipy.sparse as sp
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
from scipy.optimize import linprog
from tempfile import TemporaryDirectory
from autogluon.tabular import TabularPredictor
from autogluon.features.generators import IdentityFeatureGenerator

DATA = Path('.')                      # root folder of the raw data (searched recursively)
OUT = Path('submission_short_2.csv')
T, ETA = 168, 0.88                    # horizon (hours), hydraulic efficiency
NF = 9                                # features per unit (7 physics + 2 LP)
FOLDS, CPUS = 5, 4                    # contiguous CV folds of cases (bagging and OOF stacking features), CPUs per predictor
HP = {'GBM': {'num_boost_round': 300, 'learning_rate': 0.05, 'num_leaves': 63, 'feature_fraction': 0.5, 'bagging_fraction': 0.8, 'bagging_freq': 1},
      'CAT': {'iterations': 300, 'learning_rate': 0.05, 'depth': 6},
      'XT': {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 20, 'max_features': 0.3}}
RW = 2.0                              # weight of the cases from 2021 onwards
EPOCHS, BATCH, CH, LR, DROP, TSEEDS = 10, 32, 128, 1e-3, 0.2, 3  # temporal network
SCALES = (1, .5, 2, 4)                # price / water value scales used as augmentation
WB = 0.2                              # weight of the network in the blend
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def find(name):
    hits = sorted(DATA.rglob(name)); assert hits, f'{name} not found under {DATA.resolve()}'
    return hits[0]

## Data

In [ ]:
snake = lambda c: c.lower().replace(' ', '_')

def read(name, idx='date'):
    d = pd.read_csv(find(name)); d[idx] = pd.to_datetime(d[idx], utc=True)
    return d.set_index(idx)

# raw tables (original column names), also used by the LP
PR, IN, VO, WV = [read(f) for f in ('Historical_day_ahead_price_2015_2025.csv', 'Historical_inflow_1958_2025.csv',
                                    'Historical_volume_2015_2024.csv', 'Synthetic_water_value_2015_2024.csv')]
MF, MV = read('Constraint_min_flow.csv'), read('Constraint_min_volume.csv')
PR = PR.iloc[:, 0]
yml = yaml.safe_load(find('Tokke_Vinje_topology.yaml').read_text(encoding='utf-8')); topo = yml['model']

price = PR.to_frame('day_ahead_price')
daily = lambda d: d.rename(columns=snake).reindex(price.index, method='ffill')  # daily series -> hourly
X = (price.join(IN.rename(columns=snake).add_suffix('_inflow'))
          .join(daily(VO).add_suffix('_volume'))
          .join(daily(WV).add_suffix('_wv'))).sort_index()
Y = read('Unit_commitment_decisions.csv', 'starttime').rename(columns=snake).sort_index()
Y = Y.filter(like='result_committed').astype(np.uint8)   # drops Run No / total_calculation_time
C = pd.concat([MF, MV], axis=1).astype(np.float32)

def cases(X, index):
    """One row per case (day): 168h window of price/inflows + initial volumes + water values at start and end of the week."""
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol, wv = [[c for c in X.columns if c.endswith(s)] for s in ('_volume', '_wv')]
    pos = X.index.get_indexer(index); assert (pos >= 0).all() and (pos + T < len(X)).all()
    xh = swv(X[hourly].to_numpy(np.float32), T, axis=0)[pos].reshape(len(pos), -1)
    w = X[wv].to_numpy()
    xs = np.hstack([X[vol].to_numpy()[pos], w[pos], w[pos + T]])
    cols = [f'{c}_t{h}' for c in hourly for h in range(T)] + vol + [f'{c}_init' for c in wv] + [f'{c}_final' for c in wv]
    return pd.DataFrame(np.hstack([xh, xs]).astype(np.float32), index=index, columns=cols)

mp = pd.read_csv(find('prediction_mapping.csv'))
x_all = cases(X, Y.index)
x_kag = cases(X, pd.DatetimeIndex(pd.to_datetime(mp['starttime'], utc=True)))
print(x_all.shape, x_kag.shape, Y.shape)

## Topology and targets

In [ ]:
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
GEN = {k.lower().replace(' ', '_'): g for k, g in topo['generator'].items()}
PN = {k: g['p_nom'] for k, g in GEN.items()}
# plant -> (upstream reservoirs, downstream reservoir)
TOPO = {'vesle_kjela': (['kjelavatn'], 'foersvatn'), 'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
        'songa': (['songavatn', 'bitdalsvatn'], 'totak'), 'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
        'haukeli': (['langeidvatn', 'vatjern'], 'vinjevatn'), 'tokke': (['vinjevatn'], 'bandak'),
        'byrte': (['botnedalsvatn'], 'byrtevatn'), 'lio': (['byrtevatn'], 'bandak'), 'hogga': (['bandak'], None)}
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
WSRC = {'vinje': 'totak', 'haukeli': 'langeidvatn'}  # reservoir whose water value prices the plant's water
wsrc = lambda plant: WSRC.get(plant, TOPO[plant][0][0])
CAP_RES = {'haukeli': 'vatjern'}                     # Haukeli takes its water from Vatjern
FEED = {'vesle_kjela': ['staavatn'], 'kjela': ['langesae'], 'vinje': ['venemo', 'hyljelihyl']}  # reservoirs feeding the upstream ones
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']  # upstream first

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in Y.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
Y3 = np.stack([Y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)   # (cases, units, 168)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]

## Features: base, surrogate dispatch, constraints, unit physics

In [ ]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):  # value at t-k (edge-padded)
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):  # water level of a reservoir from its volume
    d = RES[r]['vol_head']; return np.interp(v, d['x'], d['y'])

lw = lambda a: np.log(np.clip(a, 1, None))  # scale-free log (prices / water values < 1 -> 0)

def static(X, plant):  # per-case features: volumes, log water values, filling, head
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    cols = [f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]
    A0 = X[cols].to_numpy()
    S = [np.where(np.array([not c.endswith('_volume') for c in cols]), lw(A0), A0)]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wu = X[f'{up[0]}_wv_final'].to_numpy()
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']), lw(wu) - lw(wd)]))
    return np.hstack(S)

def inflow(X, up):  # hourly inflow summed over the upstream reservoirs (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

def cap(X, plant):  # max flow of the plant (Mm3/h) from nominal power and head
    up, dn = TOPO[plant]
    r0 = CAP_RES.get(plant, up[0])
    hu = head(r0, X[f'{r0}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)

CC, _cw = list(C.columns), {}
def cwin(X):  # 168h windows of the min-flow / min-volume constraints for each case
    k = (X.index[0], len(X))
    if k not in _cw:
        pos = C.index.get_indexer(X.index); assert (pos >= 0).all()
        _cw[k] = swv(C.to_numpy(), T, axis=0)[pos]
    return _cw[k]
mvol = lambda X, rs: sum((cwin(X)[:, CC.index(r.capitalize())].max(1) for r in rs if r.capitalize() in CC), np.zeros(len(X), np.float32))
CONS = {'lio': ['r_Vest_Vassdraget', 'b_Byrtevatn', 'b_Vinjevatn', 'Byrtevatn'], 'byrte': ['b_Byrtevatn', 'Byrtevatn'], 'songa': ['Totak'],
        'vinje': ['Totak', 'b_Vinjevatn'], 'vesle_kjela': ['b_Kjelavatn'], 'kjela': ['b_Hyljelihyl'], 'tokke': ['b_Vinjevatn'], 'hogga': ['b_Bandak']}

def surr(X, plant, rel=None):
    """Greedy water-filling: pour a weekly water budget (min = no spill, max = available) into the highest-price hours."""
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant])
    src = wsrc(plant)
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s -> Mm3/h, + release of upstream plants
    Wmax = np.clip(V0 + Win - mvol(X, up), 0, T * c)
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    Wp = [np.clip((P > X[f'{src}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]
    if rel is not None: S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]]
    return S

def cons_feats(X, plant, c):
    up, dn = TOPO[plant]; n, W = len(X), cwin(X)
    bc = lambda g: np.repeat(np.asarray(g, np.float32)[:, None], T, 1)
    Vd = X[f'{dn}_volume'].to_numpy() if dn else np.zeros(n)
    Id = inflow(X, [dn]).sum(1) * 0.0036 if dn else np.zeros(n)
    F = [W[:, CC.index(k)] for k in CONS.get(plant, [])]
    F += [bc(sum(X[f'{r}_volume'].to_numpy() for r in up) - mvol(X, up)),
          bc(np.maximum(mvol(X, [dn] if dn else []) - Vd - Id, 0) / c)]
    if plant == 'lio':
        w = lambda k: W[:, CC.index(k)]
        need = np.maximum(w('r_Vest_Vassdraget') - inflow(X, ['r_vest_vassdraget']) - w('b_Vinjevatn') - w('b_Byrtevatn'), 0)
        q = (c / 0.0036)[:, None]
        F += [need, np.cumsum(need, 1) / q, bc(need.sum(1) / q[:, 0])]
    return F

def viterbi(m, s):  # m: hourly margin of being ON (cases, T); s: cost of a start + stop cycle. Returns best plan ON at t - best plan OFF at t
    h = s / 2
    Fon, Fof, Bon, Bof = m.copy(), np.zeros_like(m), np.zeros_like(m), np.zeros_like(m)
    for t in range(1, T):
        Fon[:, t] = m[:, t] + np.maximum(Fon[:, t-1], Fof[:, t-1] - h)
        Fof[:, t] = np.maximum(Fof[:, t-1], Fon[:, t-1] - h)
    for t in range(T - 2, -1, -1):
        Bon[:, t] = np.maximum(m[:, t+1] + Bon[:, t+1], Bof[:, t+1] - h)
        Bof[:, t] = np.maximum(Bof[:, t+1], m[:, t+1] + Bon[:, t+1] - h)
    return Fon + Bon - Fof - Bof

def peak(g):  # flow, reference heads and efficiencies at the peak of the turbine curves
    C_ = g['turb_eff_curves']; b = [int(np.argmax(c['y'])) for c in C_]
    return C_[0]['x'][b[0]], np.array([c['ref'] for c in C_], float), np.array([c['y'][i] / 100 for c, i in zip(C_, b)])

def unit_feats(X, plant, hmin, hmax):  # physics and start/stop costs of each unit, in commitment order
    up, dn = TOPO[plant]; pl = PL[plant]; r0 = CAP_RES.get(plant, up[0])
    P = X[PRICE].to_numpy(); w = X[f'{wsrc(plant)}_wv_final'].to_numpy()
    Hg = head(r0, X[f'{r0}_volume'].to_numpy()) - np.maximum(head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0, pl['outlet_line'])
    # threshold lam: the water value, moved only if the water budget requires it
    Ps = -np.sort(-P, 1); pk = lambda h: np.take_along_axis(Ps, np.clip(h.astype(int), 0, T - 1)[:, None], 1)[:, 0]
    lam = np.clip(w, np.where(hmax < T, pk(hmax), -np.inf), np.where(hmin > 0, pk(hmin), np.inf))
    w, lam = w[:, None], lam[:, None]
    wn, ln = np.maximum(w, 0) + 5, np.maximum(lam, 0) + 5
    F, prev = [], 0
    for j, u in enumerate(plant_units[plant]):
        g = GEN[units[u]]; q, refs, et = peak(g)
        Hj = np.maximum(Hg - float(pl['main_loss'][0]) * ((j + 1) * q)**2 - pl['penstock_loss'][g['penstock'] - 1] * q**2, 1)
        tot = 9.81e-3 * (j + 1) * q * Hj * np.interp(Hj, refs, et) * .98
        pm = np.minimum(tot - prev, g['p_max']); prev = tot  # marginal power of the (j+1)-th unit
        pref = 9.81e-3 * q * refs.max() * et.max() * .98
        rel, s = (pm / pref)[:, None], (g['startcost_const'] + g['stopcost_const']) / pref
        D1, D2, D3 = viterbi(P - w, s), viterbi(rel * P - w, s), viterbi(rel * P - lam, s)
        F += [D1 / wn, D2 / wn, D3 / ln, D1 > 0, D3 > 0, np.repeat(rel, T, 1), np.repeat(lw(lam) - lw(w), T, 1)]
    return F

## LP of the cascade
One LP per case (`scipy.optimize.linprog`): water balances, flow limits, minimum constraints. Each plant earns `price x energy - water value of its own reservoir`; each unit is a flow segment with its marginal efficiency. The per-unit features are the LP flow and the reduced cost.

In [ ]:
def make_lp():
    K, m, CON = 0.0036, topo, yml['connections']  # 1 m3/s for 1 h = 0.0036 Mm3
    RS, PS, RV, TN = list(m['reservoir']), list(m['plant']), list(m['river']), list(m['tunnel'])
    gens = {p: [g for g in m['generator'] if g.rsplit('_G', 1)[0] == p] for p in PS}
    FREE = {'Totak_Vaamarvatn', 'Bordalsvatn_Foersvatn', 'Venemo_Totak'}  # tunnels between reservoirs: flow in both directions
    INTAKE = {'Hogga': 'Bandak', 'Tokke': 'Vinjevatn', 'Lio': 'Byrtevatn', 'Byrte': 'Botnedalsvatn', 'Vinje': 'Vaamarvatn', 'Haukeli': 'Vatjern', 'Songa': 'Songavatn', 'Kjela': 'Foersvatn', 'Vesle Kjela': 'Kjelavatn'}
    OUT = {'Hogga': None, 'Tokke': 'Bandak', 'Lio': 'Bandak', 'Byrte': 'Byrtevatn', 'Vinje': 'Vinjevatn', 'Haukeli': 'Vinjevatn', 'Songa': 'Totak', 'Kjela': 'Hyljelihyl', 'Vesle Kjela': 'Foersvatn'}
    hd = lambda r, v: np.interp(v, m['reservoir'][r]['vol_head']['x'], m['reservoir'][r]['vol_head']['y'])

    def seg(p, v0):  # (marginal MW per m3/s, max flow) of each unit
        pl, dn, out, prev = m['plant'][p], OUT[p], [], 0
        Hg = hd(INTAKE[p], v0[INTAKE[p]]) - max(hd(dn, v0[dn]) if dn else 0, pl['outlet_line'])
        for j, g in enumerate(gens[p]):
            G = m['generator'][g]; q, refs, et = peak(G)
            Hj = max(Hg - float(pl['main_loss'][0]) * ((j + 1) * q)**2 - float(pl['penstock_loss'][G['penstock'] - 1]) * q**2, 1)
            tot = 9.81e-3 * (j + 1) * q * Hj * np.interp(Hj, refs, et) * .98
            out.append(((tot - prev) / q, max(G['turb_eff_curves'][0]['x']))); prev = tot
        return out

    # hourly variables: unit segments | rivers and tunnels | volumes | slacks of the minimum constraints
    segs = [(p, j) for p in PS for j in range(len(gens[p]))]; si = {s: i for i, s in enumerate(segs)}
    Fl = RV + TN; nS, nF = len(segs), len(Fl); fi = {o: nS + i for i, o in enumerate(Fl)}; ri = {r: nS + nF + i for i, r in enumerate(RS)}
    slk = list(MF.columns) + list(MV.columns); sli = {o: nS + nF + len(RS) + i for i, o in enumerate(slk)}; NV = nS + nF + len(RS) + len(slk)
    at = lambda v, t: v + t * NV
    expr = lambda o: [si[(o, j)] for j in range(len(gens[o]))] if o in m['plant'] else [fi[o]]
    inp, out = {}, {}
    for c in CON:
        a, b = c['from'], c['to']
        if c['from_type'] == 'generator': continue
        if c['to_type'] != 'reservoir' and c['from_type'] == 'reservoir': out.setdefault(a, []).append(b)
        else: inp.setdefault(b, []).append(a)
    # water balances
    R, Cc, Va, n, bal, nodes = [], [], [], 0, [], []
    def add(r, v, x): R.append(r); Cc.append(v); Va.append(x)
    for r in RS:
        for t in range(T):
            add(n, at(ri[r], t), 1.)
            if t: add(n, at(ri[r], t - 1), -1.)
            for o, sg in [(o, -K) for o in inp.get(r, [])] + [(o, K) for o in out.get(r, [])]:
                for v in expr(o): add(n, at(v, t), sg)
            bal.append((n, r, t)); n += 1
    for o in [o for o in inp if o not in m['reservoir']]:
        for t in range(T):
            for v in expr(o): add(n, at(v, t), 1.)
            for u in inp[o]:
                for v in expr(u): add(n, at(v, t), -1.)
            nodes.append((n, o, t)); n += 1
    Aeq = sp.csr_matrix((Va, (R, Cc)), shape=(n, NV * T))
    # minimum constraints with slack
    R, Cc, Va, n, ubk = [], [], [], 0, []
    for o in slk:
        for t in range(T):
            add(n, at(fi[o] if o in fi else ri[o], t), -1.); add(n, at(sli[o], t), -1.); ubk.append((o, t)); n += 1
    Aub = sp.csr_matrix((Va, (R, Cc)), shape=(n, NV * T))

    def solve(ts_):
        p0 = PR.index.get_loc(ts_); P = PR.values[p0:p0 + T]; v0 = VO.loc[ts_]
        wf = WV.reindex([ts_ + pd.Timedelta(days=7)], method='ffill').iloc[0]
        loc = lambda d: d.iloc[d.index.get_loc(ts_):][:T]
        I, mf, mv = loc(IN), loc(MF), loc(MV)
        c, lo, hi, es = np.zeros(NV * T), np.zeros(NV * T), np.full(NV * T, np.inf), {}
        for p in PS:
            S = seg(p, v0)
            for j, (e, cp) in enumerate(S):
                es[(p, j)] = e; v = at(si[(p, j)], np.arange(T))
                c[v], hi[v] = -(P * e - wf[INTAKE[p]] * S[0][0]), cp  # released water costs the water value of its own reservoir
        for o in RV:
            v = at(fi[o], np.arange(T)); ro = m['river'][o]
            c[v], hi[v] = ro.get('flow_cost_const', 2. if 'flow_cost_curve' in ro else 0.), ro.get('max_flow_const', np.inf)
        for o in TN:
            v = at(fi[o], np.arange(T)); hi[v] = 500
            if o in FREE: lo[v] = -500
        for r in RS: hi[at(ri[r], np.arange(T))] = m['reservoir'][r]['vol_head']['x'][-1]
        for o in slk: c[at(sli[o], np.arange(T))] = 1e4
        b = np.zeros(Aeq.shape[0])
        for row, r, t in bal: b[row] = K * I[r].values[t] + (float(v0[r]) if t == 0 else 0)
        for row, o, t in nodes:
            if o in I: b[row] = I[o].values[t]
        bu = np.array([-(mf[o].values[t] if o in mf else mv[o].values[t]) for o, t in ubk])
        r = linprog(c, A_ub=Aub, b_ub=bu, A_eq=Aeq, b_eq=b, bounds=np.c_[lo, hi], method='highs'); assert r.status == 0, r.message
        rc = r.lower.marginals + r.upper.marginals
        out_ = np.zeros((len(units), 2, T), np.float32)
        for p, j in segs:
            v = at(si[(p, j)], np.arange(T)); u = units.index(gens[p][j].lower().replace(' ', '_'))
            out_[u] = r.x[v] / hi[v], -rc[v] / es[(p, j)]  # relative flow, reduced cost (EUR/MWh)
        return out_
    return solve

_lp, _lps, LPK = None, {}, 1.  # LPK: scale of the reduced cost when prices are rescaled
def lp_feats(X):  # (cases, units, 2, T), each case solved once
    global _lp
    _lp = _lp or make_lp()
    for t in X.index:
        if t not in _lps: _lps[t] = _lp(t)
    return np.stack([_lps[t] for t in X.index])

## Plant features

In [ ]:
def base(X, plant, up_prob=None, rel=None):
    """Long feature matrix (cases*168 rows): common columns | NF columns per unit | S[2] (last, used by design)."""
    up, dn, n = TOPO[plant][0], TOPO[plant][1], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up); wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    rank, drank = P.argsort(1).argsort(1) / (T - 1), D.argsort(2).argsort(2).reshape(n, T) / 23
    dmean, wmean = np.repeat(D.mean(2), 24, 1), P.mean(1, keepdims=True)
    rl, sd = lw(P) - lw(wmean), P.std(1, keepdims=True) + 1  # scale-free: relative levels, z-scores, log(P/wv)
    PF = [rl, shift(rl, 1), shift(rl, -1), rank, drank, (P - dmean) / sd, (P - wmean) / sd, lw(P) - lw(wvf)]
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7] + PF + [I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:  # stacked: predicted on-probability of the upstream plants
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    dif = lambda r, w: lw(P) - lw(X[f'{r}_wv_{w}'].to_numpy()[:, None])
    F += cons_feats(X, plant, cap(X, plant)) + [dif(r, w) for r in up + ([dn] if dn else []) for w in ('init', 'final')]
    c = cap(X, plant)[:, None]
    ci = np.cumsum(I, 1) * 0.0036
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)[:, None]
    F += [ci / c, (V0 + ci - sum(RES[r]['max_vol'] for r in up)) / c]  # cumulated inflow and spill pressure
    S = surr(X, plant, rel)
    F += S
    A = [np.repeat(static(X, plant), T, 0), np.repeat(X.index.dayofyear.to_numpy(), T)]
    feed = sum((X[f'{r}_volume'].to_numpy() + inflow(X, [r]).sum(1) * 0.0036 for r in FEED.get(plant, [])), np.zeros(n)) / c[:, 0]
    U = unit_feats(X, plant, S[4][:, 0] * T, S[7][:, 0] * T + feed)  # S[4], S[7]: Wmin and Wmax as fraction of the week
    A.append(np.repeat(feed, T))
    G, wn = lp_feats(X), np.maximum(X[f'{wsrc(plant)}_wv_final'].to_numpy(), 0)[:, None] + 5
    U = [v for j, u in enumerate(plant_units[plant]) for v in U[j * 7:(j + 1) * 7] + [G[:, u, 0], G[:, u, 1] * LPK / wn]]  # + LP flow and reduced cost
    return np.column_stack([f.ravel() for f in F] + A + [f.ravel() for f in U] + [S[2].ravel()]).astype(np.float32)

## AutoGluon
One predictor per plant (units stacked with a unit-index feature), upstream plants first: LightGBM, CatBoost and ExtraTrees with fixed hyperparameters, bagged on 5 contiguous folds of cases and combined by AutoGluon's weighted ensemble. The out-of-fold probabilities feed the downstream plants; the final predictions come from the average of the fold models (no refit).

In [ ]:
ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us):  # stack the units of a plant vertically: each one gets the common columns and only its own NF columns
    r = ridx(c); R = B[r]
    s0 = R.shape[1] - 1 - NF * len(us)
    blk = lambda j, u: np.column_stack([R[:, :s0], R[:, s0 + j * NF: s0 + (j + 1) * NF], np.full(len(r), u), np.full(len(r), j), R[:, -1] - j])
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, cw):
    n, nt = len(B) // T, len(Bt) // T
    y = ylong(np.arange(n), us)
    if y.min() == y.max():
        v = float(np.clip(y[0], .01, .99))
        return np.full((len(us), n, T), v), np.full((len(us), nt, T), v)
    Xa, Xb = design(B, np.arange(n), us), design(Bt, np.arange(nt), us)
    cols = [f'f{i}' for i in range(Xa.shape[1])]
    rows = lambda a: np.tile(np.repeat(a, T), len(us))  # value per case -> rows
    d = pd.DataFrame(Xa, columns=cols).assign(y=y, fold=rows(np.arange(n) * FOLDS // n), w=rows(cw))
    with TemporaryDirectory() as p:
        m = TabularPredictor(label='y', problem_type='binary', eval_metric='roc_auc', groups='fold', sample_weight='w', path=f'{p}/m', verbosity=1).fit(
            d, hyperparameters=HP, num_stack_levels=0, dynamic_stacking=False, num_cpus=CPUS, calibrate=False,
            feature_generator=IdentityFeatureGenerator(), ag_args_ensemble={'fold_fitting_strategy': 'sequential_local'})
        print(m.leaderboard(silent=True)[['model', 'score_val', 'fit_time']].round(4).to_string(index=False), flush=True)
        oo = m.predict_proba_oof(as_multiclass=False).to_numpy().reshape(len(us), n, T)
        te = m.predict_proba(pd.DataFrame(Xb, columns=cols), as_multiclass=False).to_numpy().reshape(len(us), nt, T)
    return oo, te

def run(xtr, xte):
    cw = np.where(xtr.index.year >= 2021, RW, 1.0).astype(np.float32)
    oof, te, rtr, rte = {}, {}, {}, {}
    P = np.zeros((len(xte), len(units), T), np.float32)
    for plant in ORDER:
        us, ups = plant_units[plant], UP_PLANTS.get(plant, [])
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if ups else None
        oo, t = fit_group(base(xtr, plant, m(oof), s(rtr)), base(xte, plant, m(te), s(rte)), us, cw)
        P[:, us, :] = t.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(xtr, plant)[:, None], te[plant] * cap(xte, plant)[:, None]
        print(plant, 'ok', flush=True)
    return P

Pk_x = run(x_all, x_kag)

## Temporal network
Same features (without stacking); every case is also seen with prices and water values rescaled.

In [ ]:
def tensor(X, k=1.):  # (cases, channels, T): all features of all plants, without stacking
    global LPK
    LPK = k
    A = np.hstack([base(X, p) for p in ORDER])
    LPK = 1.
    return A.reshape(len(X), T, -1).transpose(0, 2, 1)

def scaled(X, k):  # same cases with prices and water values multiplied by k
    Z = X.copy(); cols = PRICE + [c for c in X.columns if c.endswith(('_wv_init', '_wv_final'))]
    Z[cols] = Z[cols] * k
    return Z

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.net = nn.Sequential(nn.GroupNorm(8, c), nn.Conv1d(c, c, 3, padding=d, dilation=d), nn.GELU(), nn.Dropout(DROP), nn.Conv1d(c, c, 1))
    def forward(self, x):
        return x + self.net(x)

class TCN(nn.Module):  # non-causal convolutions: the week is known in advance
    def __init__(self, nf, nu):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout1d(DROP), nn.Conv1d(nf, CH, 1), *[Block(CH, d) for d in (1, 2, 4, 8, 24, 48, 72)], nn.Conv1d(CH, nu, 1))
    def forward(self, x):
        return self.net(x)

def fit_tcn(xtr, Ytr, xp, cw):
    A = tensor(xtr); sd = A.std((0, 2), dtype=np.float64); keep = sd > 1e-4  # drop constant channels
    mu, sd = A.mean((0, 2), dtype=np.float64)[keep][None, :, None].astype(np.float32), sd[keep][None, :, None].astype(np.float32)
    tt = lambda A: torch.tensor(np.clip((A[:, keep] - mu) / sd, -8, 8), dtype=torch.float32)
    n, S = len(xtr), len(SCALES); del A
    Xtr = torch.zeros((n * S, int(keep.sum()), T), dtype=torch.float16)  # float16: half the memory for the rescaled copies
    for i, k in enumerate(SCALES): Xtr[i * n:(i + 1) * n] = tt(tensor(scaled(xtr, k), k)).half()
    Xp, Y = tt(tensor(xp)), torch.tensor(Ytr, dtype=torch.float32).repeat(S, 1, 1)
    w = torch.tensor(cw, dtype=torch.float32).repeat(S)
    def predict(m):
        m.eval()
        with torch.no_grad():
            return np.array(torch.cat([m(Xp[i:i + 128].to(DEV)).cpu() for i in range(0, len(Xp), 128)]).tolist(), dtype=np.float32)
    logits = 0
    for seed in range(TSEEDS):
        torch.manual_seed(seed)
        m = TCN(Xtr.shape[1], Y.shape[1]).to(DEV)
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-2)
        steps = EPOCHS * int(np.ceil(len(Xtr) / BATCH))
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=steps)
        for ep in range(EPOCHS):
            m.train(); tot = 0
            for b in torch.randperm(len(Xtr)).split(BATCH):
                l = (nn.functional.binary_cross_entropy_with_logits(m(Xtr[b].float().to(DEV)), Y[b].to(DEV), reduction='none').mean((1, 2)) * w[b].to(DEV)).mean()
                opt.zero_grad(); l.backward(); opt.step(); sch.step(); tot += l.item() * len(b)
            print(f'seed {seed} ep {ep + 1} loss {tot / len(Xtr):.4f}', flush=True)
        logits = logits + predict(m) / TSEEDS
    return 1 / (1 + np.exp(-logits))

Pk_t = fit_tcn(x_all, Y3, x_kag, np.where(x_all.index.year >= 2021, RW, 1.0))

## Blend and submission

In [ ]:
lg = lambda p: np.log(np.clip(p, 1e-5, 1 - 1e-5) / (1 - np.clip(p, 1e-5, 1 - 1e-5)))
P = 1 / (1 + np.exp(-((1 - WB) * lg(Pk_x) + WB * lg(Pk_t))))  # blend of the logits

In [ ]:
sub = pd.DataFrame(P.reshape(len(P), -1), columns=out_cols)[list(Y.columns)]
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
ss = pd.read_csv(find('sample_submission.csv'))
norm = {snake(c): c for c in ss.columns}
sub = sub.rename(columns=lambda c: norm.get(snake(c), c))          # column names as in sample_submission.csv
assert set(sub.columns) == set(ss.columns), set(sub.columns) ^ set(ss.columns)
sub = sub.set_index('Run No').loc[ss['Run No']].reset_index()[list(ss.columns)]
sub.to_csv(OUT, index=False)
print(sub.shape, sub.iloc[:, 1:].to_numpy().mean().round(3))